# U0.L1 — PyTorch from Zero

**Flow-Based Generative Models · UFRJ · 2026.2** · lab notebook · date: see calendar-map

By the end of this lab you own the most-reused artifact of the course: a runnable,
seeded, GPU-aware **training-loop template** (`common/train.py`), under version control.

**How this lab works** (the standard for all labs):

- Numbered **TODOs** — the instructor live-codes the *first* instance of each pattern;
  you complete the rest.
- **Checkpoint cells** every ~15 minutes: *if the assert passes, continue.*
- Solutions are released after the session.

Companion handout: the U0.L1 lecture notes on the course site.

## Part A — Environment sanity

*Reused in: every lab — the cell below is the top of every notebook in this course.*

Three jobs: prove the environment works, fix the device, seed **every** RNG stream in
sight (`random`, NumPy, and torch keep independent streams — seeding only one is the
classic way to be "seeded" and still not reproducible).

No GPU today is fine: everything in U0 runs on CPU, and the code is device-agnostic
from day one.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())


def set_seed(seed: int) -> None:
    """Seed all three RNG streams (python, numpy, torch incl. CUDA)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

**Repository layout, fixed today** — one convention for the whole course:

```
your-course-repo/
  labs/     # one notebook per lab session (this file)
  common/   # shared course code -- today's template lives here
  runs/     # checkpoints, logs (gitignored)
```

`common/` is a package, not a scratchpad: the file we build in the training-loop
part is *imported* by every later lab. U0.L2 adds to it; nothing ever rewrites it.

## Part B — Tensors and broadcasting

*Reused in: everywhere — broadcasting bugs are the #1 silent killer in
generative-model code.*

A shape bug in PyTorch usually does **not** crash: it broadcasts, produces a tensor of
a plausible shape with plausible values, and trains to convergence on the wrong loss.
Our defense, from day one: know the broadcasting rules *precisely*, and assert shapes
at boundaries.

A tensor is data plus three attributes — the trio `shape / dtype / device` answers
90% of debugging questions:

In [ ]:
x = torch.randn(4, 3)
print(x.shape, x.dtype, x.device)

# dtype: float32 is the workhorse; float64 appears today only inside the
# autograd part's gradient check. Integer tensors carry class labels.
print(torch.arange(6).dtype)

### TODO B.1 (tensors part) — the `permute`-then-`view` trap

- `view` reinterprets the same memory — **never copies**, *fails* when the layout is
  incompatible.
- `reshape` is `view` when possible, silent copy when not.
- `permute` (and `.t()`) reorders *strides*, not memory — the result is usually
  **non-contiguous**.

The cell below errors. **Read the error first**, understand *why*, then fix it.

In [ ]:
x = torch.arange(6).reshape(2, 3)
flat = x.t().view(-1)   # <- this line errors. Read the error before fixing.
# TODO(B.1): fix the line above so that flat == tensor([0, 3, 1, 4, 2, 5]).
# Two idiomatic fixes exist -- know why each one works.
print(flat)

### Broadcasting: the two rules

> 1. **Align shapes from the right.** Missing leading dimensions count as size 1.
> 2. Two aligned dimensions are compatible iff they are **equal or one of them is 1**;
>    size-1 dimensions are stretched (no copy) to match.

Broadcasting fails *loudly* only when no dimension matches — the dangerous cases are
the ones that succeed:

In [ ]:
print((torch.zeros(4, 3) + torch.zeros(3)).shape)     # rule 1: (3,) reads as (1, 3)
print((torch.zeros(4, 1) + torch.zeros(1, 3)).shape)  # rule 2: both stretch

try:
    torch.zeros(4, 3) + torch.zeros(4)                # aligned right: 3 vs 4
except RuntimeError as e:
    print("RuntimeError:", e)

### TODO B.2 (tensors part) — pairwise squared distances, no loops

For $X \in \mathbb{R}^{n \times d}$ and $Y \in \mathbb{R}^{m \times d}$, compute the
$(n, m)$ matrix $D_{ij} = \lVert x_i - y_j \rVert^2$ **without any Python loop**.

Hint: `None`-indexing inserts size-1 dimensions; the broadcast then performs the double
loop for you, vectorized.

In [ ]:
set_seed(0)
X = torch.randn(50, 3)
Y = torch.randn(40, 3)

# TODO(B.2): build D of shape (50, 40) with D[i, j] = ||X[i] - Y[j]||^2.
# No loops. Write the aligned shapes in a comment as you go.
D = ...
raise NotImplementedError("TODO B.2")

In [ ]:
# Checkpoint B -- the independent witness is the library.
assert D.shape == (50, 40)
assert torch.allclose(D, torch.cdist(X, Y) ** 2, atol=1e-5)
print("Checkpoint B passed.")

**Why this drill matters later.** The shape discipline `(batch, d)` vs `(batch, 1)`
vs `(batch,)` is exactly what makes this course's losses implementable. In U3.L1 you
will meet losses of the form

$$\mathbb{E}_{t\sim\mathrm{Unif}[0,1],\; x_1\sim q,\; x_0\sim p_0}\!\left[\, \big\lVert u_t^\theta(x_t) - (x_1 - x_0) \big\rVert^2 \,\right],$$

where `t` is a **`(batch,)` tensor** broadcast against `(batch, d)` states — one $t$
per sample, stretched across $d$ coordinates by rule 1. Course convention, fixed
today: **in every network signature, `t` is a `(B,)` float tensor with values in
$[0,1]$.**

### TODO B.3 (tensors part) — the planted device-mismatch error

The rule: **everything that touches in an operation lives on the same device** —
model *and* data, and you move both, explicitly.

On a GPU machine the cell below raises the canonical error. Diagnosis ritual: print
`.device` of every operand in the failing line and find the one that never got moved
(it is the data-loader output, roughly always). On a CPU-only machine no mismatch is
possible — read the cell anyway; the fix is the same line you will write inside every
training loop.

In [ ]:
model_b3 = nn.Linear(2, 2).to(device)
x_cpu = torch.randn(8, 2)          # born on CPU, never moved

if torch.cuda.is_available():
    out = model_b3(x_cpu)          # <- raises: read the error, then fix
else:
    print("CPU-only machine: no mismatch possible; the fix below is unchanged.")

# TODO(B.3): fix by moving the DATA to `device` (print .device of both
# operands first if the error message is not yet second nature).

## Part C — Autograd mechanics

*Reused in: U1.L1 (log-det checks), U2 (the adjoint discussion assumes you know what
autograd stores), every lab.*

In the previous session (U0.T1) we proved: **reverse-mode differentiation is right-to-left accumulation of
VJPs** along the computational graph, one cached activation per node (Proposition 1,
U0.T1). Today the same object as an API: `requires_grad=True` records operations, the
graph is built **on the fly**, `.backward()` runs the cotangent sweep into `.grad`.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
loss = x ** 2
loss.backward()
print(x.grad)   # tensor(4.)

### The accumulation demo — `.grad` adds, never overwrites

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
(x ** 2).backward(); print(x.grad)   # tensor(4.)
(x ** 2).backward(); print(x.grad)   # tensor(8.)  <- summed, not replaced!

Accumulation is a *feature* (micro-batch gradient accumulation, multi-term losses)
with a default that bites beginners. **This is why `zero_grad()` exists** — and why it
is a mandatory step of the canonical training loop.

### TODO C.1 (autograd part) — the deleted `zero_grad()`

The loop below fits $y = 3x$ but the `zero_grad()` line has been deleted.
**Predict what the loss curve does before running it.** Then run, observe, and fix.
(It does not diverge immediately — stale gradients are momentum-like at first, which
is what makes the bug so quiet.)

In [ ]:
set_seed(0)
w = torch.tensor(0.0, requires_grad=True)
xs = torch.linspace(-1, 1, 64)
ys = 3.0 * xs
opt_c1 = torch.optim.SGD([w], lr=0.1)

for step in range(100):
    loss = ((w * xs - ys) ** 2).mean()
    # TODO(C.1): one line is missing here. Which, and why does it matter?
    loss.backward()
    opt_c1.step()
    if step % 20 == 0:
        print(f"step {step:3d}  loss {float(loss.detach()):.6f}")
print("final loss:", float(loss.detach()))

### Switching the tape off — `no_grad` and `detach`

Rule of thumb: *if no `.backward()` will ever be called on it, it should not be
building a graph.* Evaluation must run under `no_grad()` (no graph → no activation
caching → less memory, faster); `detach` is the surgical, tensor-level version and
returns later in the course (EMA weights, target networks).

In [ ]:
with torch.no_grad():          # block-level: no graph built
    z = (torch.randn(3, requires_grad=True) * 2).sum()
print(z.requires_grad)         # False -- nothing was recorded

y = torch.randn(3, requires_grad=True).detach()
print(y.requires_grad)         # False -- y shares data, exits the graph

### TODO C.2 (autograd part) — the directional gradient check

Autograd is code, and code is doubted. The course-wide **gradient-check ritual**
compares it against the one estimator with no calculus and no tape — central
differences, taken along **one fixed direction** $v$ with $\lVert v\rVert = 1$:

$$\big\langle \nabla f(\theta),\, v \big\rangle \;\approx\; \frac{f(\theta + \varepsilon v) - f(\theta - \varepsilon v)}{2\varepsilon}, \qquad \lVert v \rVert = 1, \qquad \text{error } O(\varepsilon^2).$$

**Two forward evaluations, whatever the parameter count** — which is why this same
ritual still works on U0.L4's CNN and U3.L1's vector field, where a check costing two
evaluations *per parameter* would not. It is also the cost table of U0.T1 read again:
one direction is one forward-mode pass, so the right-hand side is a numerical JVP,
while autograd's side is one VJP contracted with $v$.

In [ ]:
def directional_fd(f, theta: torch.Tensor, v: torch.Tensor,
                   eps: float = 1e-6) -> float:
    """Central difference of scalar f at theta, along the unit direction v."""
    # TODO(C.2): return the central quotient
    #     (f(theta + eps * v) - f(theta - eps * v)) / (2 * eps)
    # Two forward evaluations, whatever the size of theta.
    raise NotImplementedError("TODO C.2")

**Three conditions make the comparison meaningful.** All three are part of the ritual.

**1. Run the check in `float64`.** In `float32`, machine epsilon is
$\approx 1.2\times10^{-7}$ and the subtraction in the quotient cancels
catastrophically. Measured on the model below, the identical check in `float32`
returns $d_{\mathrm{FD}} = 5.96\times10^{-2}$ against $d_{\mathrm{AD}} =
7.64\times10^{-3}$ — wrong by a factor of eight, on a gradient that is exactly right.
Train in `float32`; promote *the check*. The trick recurs in U1.L1's log-det checks.

**2. Freeze every random draw across the three evaluations.** $f(\theta)$,
$f(\theta + \varepsilon v)$ and $f(\theta - \varepsilon v)$ must see the same batch,
the same dropout mask, the same noise sample. A draw renewed between probes does not
perturb $\theta$ — it changes the function, and the quotient measures the redraw.
Here the fixture is two fixed tensors; from U0.L3 on, where the loss is genuinely
stochastic, freezing the draws is the whole difficulty.

**3. Accept on a mixed tolerance, never on a raw relative error:**

$$\lvert d_{\mathrm{AD}} - d_{\mathrm{FD}} \rvert \;\le\; \texttt{atol} + \texttt{rtol}\cdot\max\big(\lvert d_{\mathrm{AD}}\rvert,\, \lvert d_{\mathrm{FD}}\rvert\big).$$

A bare ratio $\lvert d_{\mathrm{AD}} - d_{\mathrm{FD}}\rvert / \lvert d_{\mathrm{AD}}\rvert$
is ill-conditioned exactly where the gradient is small: along a flat direction it
explodes on a correct gradient, and at zero it divides by noise.

In [ ]:
# Checkpoint C -- autograd vs. the witness, along one fixed direction.
# (nn.Sequential is a black box until the modules part -- that is fine here.)
set_seed(0)
mlp_c = nn.Sequential(
    nn.Linear(2, 16), nn.Tanh(),
    nn.Linear(16, 16), nn.Tanh(),
    nn.Linear(16, 1),
).double()                                   # float64: the check runs promoted
x_c = torch.randn(32, 2, dtype=torch.float64)   # frozen fixture: the same draws
y_c = torch.randn(32, 1, dtype=torch.float64)   # at theta, theta+eps*v, theta-eps*v
mse = nn.MSELoss()

W = mlp_c[0].weight                          # the chosen weight tensor
W0 = W.detach().clone()
g_ad = torch.autograd.grad(mse(mlp_c(x_c), y_c), W)[0].clone()

v = torch.randn_like(W0)
v /= v.norm()                                # one fixed unit direction

def loss_at(w: torch.Tensor) -> float:       # same x_c, y_c: draws frozen
    with torch.no_grad():
        W.copy_(w)
        return float(mse(mlp_c(x_c), y_c))

d_ad = float((g_ad * v).sum())
d_fd = directional_fd(loss_at, W0, v, eps=1e-6)
with torch.no_grad():
    W.copy_(W0)                              # restore after probing

atol, rtol = 1e-8, 1e-6
tol = atol + rtol * max(abs(d_ad), abs(d_fd))
print(f"d_AD = {d_ad:.10e}\nd_FD = {d_fd:.10e}\ngap  = {abs(d_ad - d_fd):.2e}  (tol {tol:.2e})")
assert abs(d_ad - d_fd) <= tol

# A check that has never failed is not known to work: it must reject a wrong gradient.
d_bad = 2.0 * d_ad
assert not abs(d_bad - d_fd) <= atol + rtol * max(abs(d_bad), abs(d_fd))
print("Checkpoint C passed (and the check rejects a doubled gradient).")

**Optional cross-check — the whole gradient, coordinate by coordinate.** For a
tensor small enough to afford it, the same construction with $v = e_i$ for every $i$
recovers the full gradient. Worth writing once, to see that the directional check is
not a weaker test but a cheaper contraction of the same one. It is a teaching aid,
never the routine check: at $O(\mathrm{numel}(x))$ forward passes it stops being
affordable at the first real network.

In [ ]:
def finite_diff_grad(f, x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Elementwise central differences -- optional; O(numel(x)) forward passes."""
    g = torch.zeros_like(x)
    flat = g.view(-1)
    for i in range(x.numel()):
        e = torch.zeros_like(x).view(-1)
        e[i] = eps
        e = e.view_as(x)
        flat[i] = (f(x + e) - f(x - e)) / (2 * eps)
    return g


g_fd_full = finite_diff_grad(loss_at, W0)
with torch.no_grad():
    W.copy_(W0)
print("contraction agrees:", abs(float((g_fd_full * v).sum()) - d_fd) < 1e-7)

> *"autograd computes exactly the VJP composition proved in U0.T1; finite differences
> is our independent witness. We will use autodiff-as-proof-assistant again in U1.L1
> (log-det via `jax.jacfwd`)."*

## Part D — `nn.Module` and optimizers

*Reused in: every model of the course.*

`nn.Module` buys three things: **parameter registration** (`model.parameters()` finds
every weight recursively — what the optimizer consumes), **device movement**
(`model.to(device)` moves everything at once), and **serialization** via `state_dict`.
(SiLU rather than ReLU for continuity with the U-Nets and DiT blocks from U0.T4
onward; nothing today depends on the choice.)

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in: int = 2, d_h: int = 64, d_out: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_h), nn.SiLU(),
            nn.Linear(d_h, d_h), nn.SiLU(),
            nn.Linear(d_h, d_out),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


model = MLP().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params} parameters, registered recursively")

### `state_dict`: the model as data

A `state_dict` is a plain dict of name → tensor: portable, inspectable. Save the
**state dict**, not the module object — a pickled module silently breaks when the
class definition changes; plain tensors do not.

In [ ]:
from pathlib import Path

Path("runs").mkdir(exist_ok=True)
torch.save(model.state_dict(), "runs/demo.pt")

model2 = MLP().to(device)
model2.load_state_dict(torch.load("runs/demo.pt", map_location=device))
same = all(
    torch.equal(a, b)
    for a, b in zip(model.state_dict().values(), model2.state_dict().values())
)
print("round trip exact:", same)
assert same

### Optimizers: black boxes with a `.step()`

Three calls: `opt.zero_grad()` clears accumulated `.grad` (autograd part!),
`loss.backward()` fills `.grad` via the VJP sweep, `opt.step()` updates parameters.
**Which optimizer and why is the next theory session's topic (U0.T2)** — today they are black boxes.

In [ ]:
opt_sgd = torch.optim.SGD(model.parameters(), lr=1e-2)
opt_adam = torch.optim.Adam(model.parameters(), lr=1e-3)
print(type(opt_sgd).__name__, "and", type(opt_adam).__name__, "instantiated.")

## Part E — The canonical training loop → `training-loop-template`

*Reused in: everything — this is the artifact.*

The loop is five steps — **forward → loss → zero_grad → backward → step** — and every
part of this lab reappears as one line of it. The template contract below is
**binding** for all later labs (they import `Trainer` and fill it):

```
common/train.py
  set_seed(seed)
  class Trainer:
    __init__(model, opt, loss_fn, device, ckpt_dir, log_every)
    fit(train_loader, val_loader, epochs)
        # loop: forward -> loss -> zero_grad -> backward -> step
    evaluate(loader)                        # runs under no_grad
    save_ckpt(tag) / load_ckpt(tag)
  minimal CSV/stdout logger
```

TODOs **E.2–E.4** live in `common/train.py` (in the student template its method bodies
are stubs; open the file next to this notebook and complete `fit`, `evaluate`, and the
checkpoint pair). The instructor live-codes `fit`'s five-step core; the checkpoint
cells below verify each piece. **E.1** is the cell after the helpers.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root, so `common` imports
from common.train import Trainer, set_seed   # same helper, now in its permanent home

**The task: two moons** — deliberately the *same dataset family* that returns in
U1.L1 (RealNVP generates it), U2.L2 (FFJORD flows it), and U3.L1 (CFM transports
noise onto it). Today: plain supervised classification, $x \in \mathbb{R}^2$,
$y \in \{0,1\}$, with the cross-entropy of U0.T1
$\widehat{R}_n(\theta) = \frac{1}{n}\sum_{i=1}^{n} \big(-\log \pi_{y_i}(x_i;\theta)\big)$.
Generator and plotting helper are given — today is not a matplotlib lab.

In [ ]:
def two_moons(n: int, noise: float = 0.08,
              generator: torch.Generator | None = None):
    """n points on two interleaved half-circles, labels in {0, 1}."""
    half = n // 2
    t0 = torch.rand(half, generator=generator) * torch.pi
    t1 = torch.rand(n - half, generator=generator) * torch.pi
    x0 = torch.stack([torch.cos(t0), torch.sin(t0)], dim=1)
    x1 = torch.stack([1.0 - torch.cos(t1), 0.5 - torch.sin(t1)], dim=1)
    x = torch.cat([x0, x1]) + noise * torch.randn(n, 2, generator=generator)
    y = torch.cat([torch.zeros(half), torch.ones(n - half)]).long()
    return x, y


def plot_decision_boundary(model: nn.Module, x: torch.Tensor, y: torch.Tensor,
                           device: str) -> None:
    """Class-1 probability shading + the training points. Runs under no_grad."""
    xg = torch.linspace(-1.6, 2.6, 200)
    yg = torch.linspace(-1.1, 1.6, 200)
    gx, gy = torch.meshgrid(xg, yg, indexing="xy")
    grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1).to(device)
    model.eval()
    with torch.no_grad():
        p1 = torch.softmax(model(grid), dim=1)[:, 1].reshape(gx.shape).cpu()
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.contourf(gx, gy, p1, levels=21, cmap="coolwarm", alpha=0.5)
    ax.contour(gx, gy, p1, levels=[0.5], colors="k", linewidths=1.2)
    ax.scatter(x[:, 0], x[:, 1], c=y, s=7, cmap="coolwarm", alpha=0.8)
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
    plt.show()

### TODO E.1 (training-loop part) — data and loaders

Build `train_loader` and `val_loader`: 1000 training and 400 validation points from
`two_moons`, batch size 128, **seeded** shuffling (a `torch.Generator` for the data
and one for the loader — reproducibility is baked in from day one).

In [ ]:
set_seed(0)
gen = torch.Generator().manual_seed(0)

# TODO(E.1): x_train, y_train (n=1000) and x_val, y_val (n=400) from two_moons,
# then TensorDataset + DataLoader (batch_size=128; shuffle the TRAIN loader
# with a seeded generator; no shuffle for val).
train_loader = ...
val_loader = ...
raise NotImplementedError("TODO E.1")

### Checkpoint E.2 — `fit` trains (loss decreases)

In [ ]:
set_seed(0)
model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()              # takes LOGITS, not probabilities

trainer = Trainer(model, opt, loss_fn, device, ckpt_dir="runs", log_every=10)
history = trainer.fit(train_loader, val_loader, epochs=60)

losses = history["train_loss"]
assert losses[-1] < 0.15 < losses[0]         # the loss decreased
print("Checkpoint E.2 passed.")

### Checkpoint E.3 — `evaluate` runs under `no_grad` and the boundary is real

In [ ]:
val = trainer.evaluate(val_loader)
print(val)
assert val["acc"] > 0.95                     # > 95% val accuracy on two moons
print("Checkpoint E.3 passed.")

### Checkpoint E.4 — the checkpoint round trip is exact

In [ ]:
trainer.save_ckpt("final")

fresh = Trainer(MLP(), torch.optim.Adam(MLP().parameters(), lr=1e-3),
                loss_fn, device, ckpt_dir="runs")
fresh.load_ckpt("final")

same = all(
    torch.equal(a, b)
    for a, b in zip(trainer.model.state_dict().values(),
                    fresh.model.state_dict().values())
)
assert same
print("Checkpoint E.4 passed: save -> load -> identical weights.")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.semilogy(range(1, len(losses) + 1), losses)
ax.set_xlabel("epoch"); ax.set_ylabel(r"training loss $\widehat{R}_n(\theta)$")
plt.show()

plot_decision_boundary(trainer.model, x_train, y_train, device)

## Part F — Freeze and close

**Commit `common/train.py` now** — this is the lab's final checkpoint:

```bash
git add common/train.py
git commit -m "training-loop-template v1"
```

U0.L2 imports this exact file and *adds* the hygiene stack (logging, schedules,
checkpoint discipline); nothing gets rewritten. If your template diverges from the
contract above, later lab scaffolds will not import cleanly — the contract is the
interface.

**Homework-lite (unmarked, ~10 min).** Re-run the training with seeds 0, 1, 2 and
*eyeball* the three final validation accuracies — no statistics required, just notice
the spread is not zero. That spread is the next theory session's opening question.

In [ ]:
for seed in (0, 1, 2):
    set_seed(seed)
    m = MLP().to(device)
    t = Trainer(m, torch.optim.Adam(m.parameters(), lr=1e-3), loss_fn,
                device, ckpt_dir="runs", log_every=1000)
    t.fit(train_loader, val_loader, epochs=60)
    print(f"seed {seed}: val acc {t.evaluate(val_loader)['acc']:.3f}")

**Next.** The next theory session (U0.T2): making training *work* — the optimizer
black boxes get opened, plus learning-rate schedules, initialization, normalization.
The lab after it (U0.L2): the hygiene stack — your template grows logging, schedules, and
reproducibility discipline. Bring the committed template to both.